# BERT Mask Filling

Choose a sentence. How does its context change the top candidates?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/google-bert/bert-base-uncased

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import tarfile
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    BertTokenizer, BertModel, BertForMaskedLM,
    CLIPProcessor, CLIPModel, CLIPTokenizerFast,
)
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
BERT_ID = 'bert-base-uncased'

print(f"📦 Loading BERT: {BERT_ID}")
bert_tokenizer = BertTokenizer.from_pretrained(BERT_ID)
bert_model = BertModel.from_pretrained(BERT_ID).to(device)
bert_model.eval()

bert_mlm = BertForMaskedLM.from_pretrained(BERT_ID).to(device)
bert_mlm.eval()

params = sum(p.numel() for p in bert_model.parameters())
print(f"   ✅ Loaded!")
print(f"   - Parameters: {params / 1e6:.1f}M")
print(f"   - Hidden size: {bert_model.config.hidden_size}")
print(f"   - Layers: {bert_model.config.num_hidden_layers}")
print(f"   - Attention heads: {bert_model.config.num_attention_heads}")
print(f"   - Vocab size: {bert_tokenizer.vocab_size}")

In [ ]:
@torch.no_grad()
def predict_masked(text, top_k=10):
    """
    Predict the word that fits the position of the [MASK] token.
    """
    inputs = bert_tokenizer(text, return_tensors='pt').to(device)
    outputs = bert_mlm(**inputs)
    logits = outputs.logits  # (1, seq_len, vocab_size)

    # Find the [MASK] positions
    mask_idx = (inputs['input_ids'][0] == bert_tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

    results = []
    for idx in mask_idx:
        mask_logits = logits[0, idx]
        probs = torch.softmax(mask_logits, dim=0)
        top_probs, top_indices = probs.topk(top_k)

        predictions = []
        for prob, token_id in zip(top_probs, top_indices):
            word = bert_tokenizer.decode(token_id.item())
            predictions.append((word, prob.item()))
        results.append(predictions)

    return results

# MLM demo
mlm_examples = [
    "The [MASK] is wagging its tail happily.",
    "A [MASK] is sleeping on the windowsill.",
    "The golden [MASK] is the most popular dog breed.",
    "My pet [MASK] loves to chase mice.",
    "[MASK] are known for their loyalty to humans.",
    "The [MASK] purrs when you stroke its fur.",
]

print("🎭 Masked Language Modeling (BERT)\n")
print("BERT predicts the word that fills [MASK]:\n")

for text in mlm_examples:
    preds = predict_masked(text, top_k=5)
    print(f"  📝 \"{text}\"")
    for mask_preds in preds:
        pred_str = ", ".join([f"{w} ({p:.1%})" for w, p in mask_preds])
        print(f"     → {pred_str}")
    print()

## Demo code

Choose a sentence. How does its context change the top candidates?

In [ ]:
sentence = "The [MASK] is wagging its tail happily."
candidates = predict_masked(sentence, top_k=5)
for word, probability in candidates[0]:
    print(f"{word:12s} {probability:.3f}")

In [ ]:
# Visualize MLM predictions as bar charts
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🎭 BERT Masked Language Model Predictions', fontsize=16, fontweight='bold')

colors = ['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

for i, text in enumerate(mlm_examples):
    ax = axes[i // 3, i % 3]
    preds = predict_masked(text, top_k=7)[0]

    words = [p[0] for p in preds]
    probs = [p[1] for p in preds]

    bars = ax.barh(range(len(words)), probs, color=colors[i], alpha=0.8, edgecolor='black')
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=11)
    ax.set_xlabel('Probability')
    ax.set_title(text.replace('[MASK]', '____'), fontsize=10, fontweight='bold')
    ax.invert_yaxis()

    for bar, prob in zip(bars, probs):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
               f'{prob:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Interpretation

Probabilities are conditioned on the supplied sentence.

Compare the output with the evidence in the input. A generated answer can be wrong.